In [1]:
import numpy as np
from scipy import stats

def compare_session_length(n1, mean1, std1,
                           n2, mean2, std2,
                           alpha=0.05):
    """
    Welch's two-sample t-test comparing mean session length.

    Parameters:
        n1, mean1, std1 : iOS group size, mean, std dev (minutes)
        n2, mean2, std2 : Android group size, mean, std dev (minutes)
        alpha           : significance level

    Returns a dict with:
        - mean_diff   : mean1 - mean2
        - se_diff     : standard error of the difference
        - t_stat      : test statistic
        - df          : Welch's degrees of freedom
        - p_value     : two-sided p-value
        - ci_95       : (lower, upper) 95% CI on mean_diff
        - reject_null : True if p_value < alpha
        - conclusion  : 'iOS sessions significantly longer'
                        or 'No significant difference'
    """
    # ── Input validation ──────────────────────────────────
    assert n1 > 0 and n2 > 0,        "sample sizes must be positive"
    assert std1 >= 0 and std2 >= 0,  "std devs must be non-negative"
    assert 0 < alpha < 1,            "alpha must be between 0 and 1"

    # ── Step 1: Variance components ───────────────────────
    se1 = std1**2 / n1          # variance contribution from group 1
    se2 = std2**2 / n2          # variance contribution from group 2

    # ── Step 2: Standard error of the difference ──────────
    se_diff = np.sqrt(se1 + se2)

    # ── Step 3: Test statistic ────────────────────────────
    mean_diff = mean1 - mean2
    t_stat    = mean_diff / se_diff

    # ── Step 4: Welch's degrees of freedom ───────────────
    df = (se1 + se2)**2 / (se1**2 / (n1 - 1) + se2**2 / (n2 - 1))

    # ── Step 5: Two-sided p-value ─────────────────────────
    # abs(t_stat) → area in right tail → ×2 for both tails
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df))

    # ── Step 6: 95% CI on mean difference ────────────────
    # t_crit × se_diff = margin of error (same units as mean_diff)
    t_crit   = stats.t.ppf(0.975, df)
    ci_lower = mean_diff - t_crit * se_diff
    ci_upper = mean_diff + t_crit * se_diff

    # ── Step 7: Decision ──────────────────────────────────
    reject_null = p_value < alpha
    conclusion  = ('iOS sessions significantly longer'
                   if reject_null else 'No significant difference')

    return {
        'mean_diff'  : mean_diff,
        'se_diff'    : se_diff,
        't_stat'     : t_stat,
        'df'         : df,
        'p_value'    : p_value,
        'ci_95'      : (ci_lower, ci_upper),
        'reject_null': reject_null,
        'conclusion' : conclusion
    }

# ── Output validation ─────────────────────────────────
    assert se_diff >= 0,                       "se_diff must be non-negative"
    assert df > 0,                             "degrees of freedom must be positive"
    assert 0 <= p_value <= 1,                  "p_value must be in [0, 1]"
    assert ci_lower < ci_upper,                "CI lower must be less than upper"
    assert reject_null == (p_value < alpha),   "reject_null must match p_value < alpha"
    assert conclusion in (
        'iOS sessions significantly longer',
        'No significant difference'
    ),                                         "conclusion must be one of the two valid strings"

# ── Run it ────────────────────────────────────────────────
result = compare_session_length(
    n1=1200, mean1=18.4, std1=6.2,   # iOS
    n2=1500, mean2=16.9, std2=7.8,   # Android
    alpha=0.05
)

for k, v in result.items():
    print(f"{k:>12}: {v}")

   mean_diff: 1.5
     se_diff: 0.2694315002618167
      t_stat: 5.567277762779755
          df: 2697.892798274878
     p_value: 2.842469193353736e-08
       ci_95: (np.float64(0.9716869461456848), np.float64(2.028313053854315))
 reject_null: True
  conclusion: iOS sessions significantly longer
